In [0]:
%pip install yfinance
%pip install pandas

In [0]:
import yfinance as yf
import pandas as pd

from datetime import timedelta

from delta.tables import DeltaTable

from pyspark.sql.functions import (
    col,
    current_timestamp,
    max as spark_max
)

from pyspark.sql.types import (
    StructType,
    StructField,
    DateType,
    StringType,
    TimestampType
)

# ==========================================================
# Configuration
# ==========================================================

DATABASE = "finance_intelligence_hub.bronze"
TABLE_NAME = "stock_prices_raw"
FULL_TABLE_NAME = f"{DATABASE}.{TABLE_NAME}"

# ==========================================================
# Get all tickers
# ==========================================================

companies_row_count = spark.table(f"{DATABASE}.companies").count()

tickers_raw = (
    spark.table(f"{DATABASE}.companies")
    .select("ticker")
    .toPandas()["ticker"]
)

null_ticker_count = tickers_raw.isna().sum()

tickers = (
    tickers_raw
    .dropna()
    .drop_duplicates()
    .tolist()
)

duplicate_ticker_count = (
    companies_row_count - null_ticker_count - len(tickers)
)

print("=" * 60)
print(f"companies table row count   : {companies_row_count}")
print(f"Null tickers               : {null_ticker_count}")
print(f"Duplicate tickers          : {duplicate_ticker_count}")
print(f"Distinct usable tickers    : {len(tickers)}")
print("=" * 60)

if len(tickers) != companies_row_count:
    print(
        f"NOTE: {companies_row_count - len(tickers)} companies rows are not "
        f"represented as a distinct ticker (nulls and/or duplicates). "
        f"The pipeline will process {len(tickers)} unique tickers."
    )
else:
    print("All companies rows map to a distinct ticker. Nothing dropped.")

# ==========================================================
# Check table exists
# ==========================================================

tables = (
    spark.sql(f"SHOW TABLES IN {DATABASE}")
    .select("tableName")
    .toPandas()["tableName"]
    .tolist()
)

table_exists = TABLE_NAME in tables

# ==========================================================
# Get last loaded date PER TICKER
# ==========================================================

DEFAULT_START_DATE = "2022-01-01"

if table_exists:

    ticker_last_dates = (
        spark.table(FULL_TABLE_NAME)
        .groupBy("ticker")
        .agg(spark_max("date").alias("max_date"))
        .toPandas()
    )

    start_date_by_ticker = {
        row["ticker"].strip().upper(): row["max_date"].strftime("%Y-%m-%d")
        for _, row in ticker_last_dates.iterrows()
        if row["max_date"] is not None
    }

else:

    start_date_by_ticker = {}

print(f"Tickers with existing history: {len(start_date_by_ticker)}")
print(f"New tickers will start from: {DEFAULT_START_DATE}")

# ==========================================================
# Identify missing tickers (in companies, not yet in stock_prices_raw)
# Normalize (strip whitespace, uppercase) so casing/whitespace
# differences don't create false "missing" or duplicate tickers
# ==========================================================

def normalize(t):
    return t.strip().upper()

all_tickers_norm = {normalize(t): t for t in tickers}
loaded_tickers_norm = {normalize(t) for t in start_date_by_ticker.keys()}

missing_norm = set(all_tickers_norm.keys()) - loaded_tickers_norm
existing_norm = set(all_tickers_norm.keys()) & loaded_tickers_norm

pending_tickers = [all_tickers_norm[t] for t in missing_norm]
loaded_tickers = [all_tickers_norm[t] for t in existing_norm]

print("=" * 60)
print(f"Total tickers in companies table : {len(tickers)}")
print(f"Already in stock_prices_raw     : {len(loaded_tickers)}")
print(f"Missing (never loaded)          : {len(pending_tickers)}")
print("=" * 60)
print("Sample of missing tickers:", pending_tickers[:10])

assert len(pending_tickers) + len(loaded_tickers) == len(all_tickers_norm), \
    "Mismatch: pending + loaded should equal total unique tickers"

# ==========================================================
# Prioritize missing tickers first, then already-loaded ones
# ==========================================================

tickers = pending_tickers + loaded_tickers

print(f"Processing order: {len(pending_tickers)} missing tickers first, "
      f"then {len(loaded_tickers)} already-loaded tickers")

# ==========================================================
# Create empty table if first run
# ==========================================================

if not table_exists:

    empty_schema = spark.createDataFrame(
        [],
        StructType([
            StructField("date", DateType(), True),
            StructField("adj_close", StringType(), True),
            StructField("close", StringType(), True),
            StructField("high", StringType(), True),
            StructField("low", StringType(), True),
            StructField("open", StringType(), True),
            StructField("volume", StringType(), True),
            StructField("ticker", StringType(), True),
            StructField("dwh_loaded_at", TimestampType(), True),
        ])
    )

    (
        empty_schema.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(FULL_TABLE_NAME)
    )

    table_exists = True

    print("Table created.")

# ==========================================================
# Group tickers into batches by shared start_date
# (tickers needing the same start_date can be downloaded
# together in a single yfinance call)
# ==========================================================

from collections import defaultdict

BATCH_SIZE = 100

cohorts = defaultdict(list)

for t in tickers:
    t_start = start_date_by_ticker.get(normalize(t), DEFAULT_START_DATE)
    cohorts[t_start].append(t)

batches = []

for start_date, batch_tickers in cohorts.items():
    for i in range(0, len(batch_tickers), BATCH_SIZE):
        batches.append(
            (start_date, batch_tickers[i:i + BATCH_SIZE])
        )

print(f"Total tickers  : {len(tickers)}")
print(f"Total batches  : {len(batches)} (batch size = {BATCH_SIZE})")

# ==========================================================
# Download + Merge one batch at a time
# ==========================================================

BRONZE_COLUMNS = [
    "date", "adj_close", "close", "high", "low", "open", "volume", "ticker"
]

RENAME_MAP = {
    "Date": "date",
    "Adj_Close": "adj_close",
    "Close": "close",
    "High": "high",
    "Low": "low",
    "Open": "open",
    "Volume": "volume"
}

def clean_ticker_frame(sub, ticker):
    sub = sub.reset_index()
    sub.columns = (
        sub.columns
        .str.strip()
        .str.replace(" ", "_")
    )
    sub = sub.rename(columns=RENAME_MAP)
    sub["ticker"] = ticker
    return sub

total_inserted = 0
total_tickers_processed = 0
total_tickers_failed = 0

for batch_num, (start_date, batch_tickers) in enumerate(batches, start=1):

    print(
        f"[Batch {batch_num}/{len(batches)}] "
        f"{len(batch_tickers)} tickers from {start_date}"
    )

    try:

        raw = yf.download(
            batch_tickers,
            start=start_date,
            auto_adjust=False,
            progress=False,
            threads=True,
            group_by="ticker"
        )

        if raw.empty:
            print("  (no data returned for this batch)")
            continue

        frames = []

        if isinstance(raw.columns, pd.MultiIndex):

            available_tickers = set(raw.columns.get_level_values(0))

            for ticker in batch_tickers:

                if ticker not in available_tickers:
                    total_tickers_failed += 1
                    continue

                sub = raw[ticker].dropna(how="all")

                if sub.empty:
                    continue

                frames.append(clean_ticker_frame(sub, ticker))

        else:

            # Only one ticker ended up in this batch
            sub = raw.dropna(how="all")

            if not sub.empty:
                frames.append(clean_ticker_frame(sub, batch_tickers[0]))

        if not frames:
            print("  (no usable rows in this batch)")
            continue

        batch_df = pd.concat(frames, ignore_index=True)

        # Keep only new dates (safety net beyond yfinance's own start filter)
        batch_df = batch_df[
            pd.to_datetime(batch_df["date"]) >=
            pd.to_datetime(start_date)
        ]

        # Drop rows with no actual price data (e.g. delisted/halted tickers)
        batch_df = batch_df.dropna(subset=["close"])

        if batch_df.empty:
            continue

        batch_df = batch_df[BRONZE_COLUMNS]

        # Convert all to string (Bronze Layer)
        for c in ["adj_close", "close", "high", "low", "open", "volume"]:
            batch_df[c] = batch_df[c].astype(str)

        spark_df = spark.createDataFrame(batch_df)

        spark_df = (
            spark_df
            .withColumn("date", col("date").cast("date"))
            .withColumn("adj_close", col("adj_close").cast("string"))
            .withColumn("close", col("close").cast("string"))
            .withColumn("high", col("high").cast("string"))
            .withColumn("low", col("low").cast("string"))
            .withColumn("open", col("open").cast("string"))
            .withColumn("volume", col("volume").cast("string"))
            .withColumn("ticker", col("ticker").cast("string"))
            .withColumn("dwh_loaded_at", current_timestamp())
        )

        spark_df.createOrReplaceTempView("tmp_stock_prices_batch")

        delta_table = DeltaTable.forName(spark, FULL_TABLE_NAME)

        (
            delta_table.alias("target")
            .merge(
                spark.table("tmp_stock_prices_batch").alias("source"),
                """
                target.ticker = source.ticker
                AND
                target.date = source.date
                """
            )
            .whenNotMatchedInsert(
                values={
                    "date": "source.date",
                    "adj_close": "source.adj_close",
                    "close": "source.close",
                    "high": "source.high",
                    "low": "source.low",
                    "open": "source.open",
                    "volume": "source.volume",
                    "ticker": "source.ticker",
                    "dwh_loaded_at": "source.dwh_loaded_at"
                }
            )
            .execute()
        )

        inserted = spark_df.count()
        total_inserted += inserted
        total_tickers_processed += len(frames)

        print(
            f"  ✓ {len(frames)} tickers processed "
            f"({inserted} rows, {len(batch_tickers) - len(frames)} skipped/empty)"
        )

    except Exception as e:

        total_tickers_failed += len(batch_tickers)

        print(f"  ✗ Error with batch {batch_num}: {e}")

print("=" * 60)
print("Pipeline Finished")
print(f"Tickers targeted   : {len(tickers)}")
print(f"Tickers processed  : {total_tickers_processed}")
print(f"Tickers failed     : {total_tickers_failed}")
print(f"Rows Read          : {total_inserted}")
print("=" * 60)
